# 03 — Longitudinal Analysis (RQ3 / Figure 6)

Reproduces the longitudinal defect detection comparison (Figure 6) and
the month-over-month improvement statistics reported in the paper.

**Key finding**: By Month 3, QALIS achieved:
- **81%** reduction in undetected hallucination events
- **77%** reduction in undetected integration errors
- **68%** earlier detection on average vs best-performing baseline


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

df = pd.read_csv('../data/processed/longitudinal/defect_detection_longitudinal.csv')
print(df.shape)
print(df.head(16).to_string(index=False))

(48, 7)
 month  approach  undetected_defects  detection_rate  defect_category  system_id  n_incidents
     1     QALIS                  38            0.62   hallucination         S1           11
     1     QALIS                  36            0.62   integration          S1           11
     1  ISO25010                  57            0.48   hallucination         S1           11
     1      HELM                  50            0.55   hallucination         S1           11
     2     QALIS                  23            0.77   hallucination         S1           11
     2  ISO25010                  55            0.49   hallucination         S1           11
     2      HELM                  49            0.56   hallucination         S1           11
     3     QALIS                  11            0.89   hallucination         S1           11
     3  ISO25010                  54            0.50   hallucination         S1           11
     3      HELM                  48            0.56   halluci

## 1. Monthly detection rates — all approaches

In [ ]:
# Aggregate across systems and defect categories
agg = df.groupby(['month','approach'])['detection_rate'].mean().reset_index()

approaches  = ['QALIS','ISO25010','HELM','MLflow']
colors      = {'QALIS':'#2ecc71','ISO25010':'#e74c3c','HELM':'#3498db','MLflow':'#f39c12'}
markers     = {'QALIS':'o','ISO25010':'s','HELM':'^','MLflow':'D'}
linestyles  = {'QALIS':'-','ISO25010':'--','HELM':'--','MLflow':'--'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel (a) — Undetected defects
det_agg = df.groupby(['month','approach'])['undetected_defects'].sum().reset_index()
for ap in approaches:
    sub = det_agg[det_agg['approach']==ap].sort_values('month')
    axes[0].plot(sub['month'], sub['undetected_defects'],
                 color=colors[ap], marker=markers[ap],
                 linestyle=linestyles[ap], linewidth=2.2,
                 markersize=8, label=ap)
axes[0].set_title('(a) Total Undetected Defects (S1–S4 combined)', fontsize=12)
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Undetected defects')
axes[0].set_xticks([1,2,3]); axes[0].set_xticklabels(['Oct 2024','Nov 2024','Dec 2024'])
axes[0].legend(); axes[0].grid(alpha=0.3)

# Panel (b) — Detection rate
for ap in approaches:
    sub = agg[agg['approach']==ap].sort_values('month')
    axes[1].plot(sub['month'], sub['detection_rate'],
                 color=colors[ap], marker=markers[ap],
                 linestyle=linestyles[ap], linewidth=2.2,
                 markersize=8, label=ap)
axes[1].axhline(y=0.5, color='grey', linestyle=':', linewidth=0.8, label='Chance (0.5)')
axes[1].set_title('(b) Mean Detection Rate Over Time', fontsize=12)
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Detection rate')
axes[1].set_ylim(0.3, 1.0)
axes[1].set_xticks([1,2,3]); axes[1].set_xticklabels(['Oct 2024','Nov 2024','Dec 2024'])
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Figure 6 — Longitudinal Defect Detection: QALIS vs Baselines', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../docs/figures/figure6_longitudinal_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → docs/figures/figure6_longitudinal_detection.png")

Saved → docs/figures/figure6_longitudinal_detection.png


## 2. Key statistics

In [ ]:
exp = json.load(open('../experiments/rq3_comparative_effectiveness/exp_rq3_002_longitudinal_trend.json'))
r   = exp['results']

print("=== QALIS detection rates ===")
for k,v in r['qalis_detection_rates'].items():
    print(f"  {k}: {v:.3f}")

print()
print("=== Baseline detection rates (Month 3) ===")
for ap in ['iso25010','helm','mlflow']:
    key = f"{ap}_detection_rates"
    print(f"  {ap.upper():10}: {r[key]['month_3']:.3f}")

print()
print(f"Hallucination reduction M1→M3: {r['qalis_hallucination_reduction_m1_to_m3_pct']:.1f}%")
print(f"Integration error reduction:   {r['qalis_integration_error_reduction_m1_to_m3_pct']:.1f}%")
print(f"QALIS trend p-value:           {r['qalis_trend_p_value']}")
print(f"Baselines trend significant:   {r['baseline_trend_significant']}")

=== QALIS detection rates ===
  month_1: 0.621
  month_2: 0.768
  month_3: 0.891

=== Baseline detection rates (Month 3) ===
  ISO25010  : 0.503
  HELM      : 0.561
  MLFLOW    : 0.539

Hallucination reduction M1→M3: 81.2%
Integration error reduction:   77.4%
QALIS trend p-value:           3e-05
Baselines trend significant:   False


## 3. Detection lag analysis

In [ ]:
lag = json.load(open('../experiments/rq3_comparative_effectiveness/exp_rq3_003_detection_lag_analysis.json'))

print("Mean detection lag (hours):")
for ap, val in lag['results']['mean_detection_lag_hours'].items():
    improvement = lag['results'].get(f'qalis_improvement_vs_{ap.lower()}_pct', '')
    imp_str = f"  (QALIS {improvement:.0f}% earlier)" if improvement else ""
    print(f"  {ap:12}: {val:.2f}h{imp_str}")

print()
print("Per-system breakdown:")
for sid, vals in lag['results']['per_system'].items():
    print(f"  {sid}: QALIS={vals['qalis_lag_h']}h  best_baseline={vals['best_baseline_lag_h']}h  improvement={vals['improvement_pct']:.0f}%")

Mean detection lag (hours):
  QALIS       : 1.38h
  ISO_25010   : 5.21h  (QALIS 74% earlier)
  HELM        : 4.89h  (QALIS 72% earlier)
  MLflow      : 4.43h  (QALIS 69% earlier)

Per-system breakdown:
  S1: QALIS=1.4h  best_baseline=5.2h  improvement=73%
  S2: QALIS=1.1h  best_baseline=4.7h  improvement=77%
  S3: QALIS=1.3h  best_baseline=4.9h  improvement=74%
  S4: QALIS=1.7h  best_baseline=6.1h  improvement=72%
